# Cell Surface Visualization

This notebook provides tools to visualize `.ply` surfaces of neurons as rotating 3D objects.

In [ ]:
# Reload modules if changed outside this script
%load_ext autoreload
%autoreload 2

In [ ]:
import pyvista as pv
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, Image, HTML
from multiprocessing import Pool, cpu_count
import ipywidgets as widgets
import numpy as np

import sys; sys.path.insert(0, '..')
from scripts.visualize_neurons import get_cell_ply_files, create_rotation_gif, update_interactive_plotter, create_combined_gif

In [ ]:
# Path to the cell surfaces
surfaces_directory = Path("../emimesh/results/cells_lowres/")

In [ ]:
# Print numer of cells
print(f'Number of astrocytes: {len(list(surfaces_directory.glob("astrocytes/*")))}')
print(f'Number of microglias: {len(list(surfaces_directory.glob("microglias/*")))}')
print(f'Number of neurons: {len(list(surfaces_directory.glob("neurons/*")))}')

In [ ]:

# Assumes surface_directory/cell_0, surface_directory/cell_1, ...
# surface_paths = [Path(p) for p in get_cell_ply_files(surfaces_directory)]
# surface_paths = sorted(surface_paths, key=lambda x: int(x.parent.parent.name.split("_")[1]))

# Assumes surface_directory/cell_type/cell_0, surface_directory/cell_type/cell_1, ...
surface_paths = [Path(p) for p in get_cell_ply_files(surfaces_directory)]
surface_paths = sorted(surface_paths, key=lambda x: int(x.parent.parent.name.split("_")[-1]))


print(f"Found {len(surface_paths)} cell surfaces.")
print(surface_paths[:3])

## GIF Generator

This function creates a rotating 3D animation of the surface and saves it as a GIF.

In [ ]:
i = 0
for gif in surfaces_directory.rglob("*.gif"):
    if "gifs" in gif.parts:
        continue
    gif.unlink()
    i += 1
print(f'Removed {i} .gif files')

In [ ]:
# Generate GIF for the first neuron
gif_path = create_rotation_gif((surface_paths[0], 7), frames_per_degree=1.0, verbose=True)

# Show the generated GIF
display(Image(filename=gif_path))

### Batch Processing

Loop through all neurons in the results folder.

In [ ]:
print(f"Using {cpu_count()//2} CPU cores for parallel processing.")
with Pool(processes=cpu_count()//2) as pool:
    tasks = [(path, idx) for idx, path in enumerate(surface_paths)]
    results = list(tqdm(pool.imap(create_rotation_gif, tasks), total=len(tasks), desc="Generating GIFs", colour="blue"))

print(f"Generated GIFs for {len(results)} neuron surfaces.")

## Display all cells

In [ ]:
import base64
start_idx = 16 * 2
max_N = 16

# Collect all surface.gif paths
gif_paths = [str(p.parent / "surface.gif") for p in surface_paths if (p.parent / "surface.gif").exists()]
neuron_labels = {path: p.parent.parent.name for path, p in zip(gif_paths, surface_paths)}


html_str = '<div style="display: grid; grid-template-columns: repeat(auto-fill, minmax(500px, 1fr)); gap: 15px; text-align: center;">'
for path in gif_paths[start_idx:start_idx + max_N]:
    label = neuron_labels[path].replace("_", " ")
    
    # 1. Read the GIF file binary data and convert it to a Base64 string
    with open(path, "rb") as f:
        encoded_gif = base64.b64encode(f.read()).decode("utf-8")
    
    html_str += '<div style="border: 1px solid #ddd; padding: 5px; border-radius: 5px; background-color: #111;">'
    html_str += f'<img src="data:image/gif;base64,{encoded_gif}" style="width: 100%; height: auto;"><br>'
    html_str += f'<strong style="color: white; font-size: 12px;">{label}</strong>'
    html_str += '</div>'

html_str += '</div>'

HTML(html_str)

## Combine N cells into a grid GIF

In [ ]:

for cell_type in ["astrocytes", "microglias", "neurons", "all"]:
    # Configuration & Downsampling Settings
    start_idx = 0

    # Collect all surface.gif paths for the specified cell type
    if cell_type != "all":
        gif_paths = [p.parent / "surface.gif" for p in surface_paths if (p.parent / "surface.gif").exists() and cell_type in str(p)]
    else:
        gif_paths = [p.parent / "surface.gif" for p in surface_paths if (p.parent / "surface.gif").exists()] # All

    for N in np.arange(2, 20, 4)**2:
        output_gif_path = create_combined_gif(
            gif_paths=gif_paths,
            output_path=gif_paths[0].parent.parent.parent.parent / "gifs" /  f"{cell_type}_{np.sqrt(N):.0f}x{np.sqrt(N):.0f}.gif",
            N=N,
            start_idx=start_idx,
            # resize_factor=1 / np.sqrt(N),
            resize_factor=1 / N**(1/4),
            frame_step=1,
            duration=1.0,
            verbal=False
        )

    # Display the combined GIF
    display(Image(filename=output_gif_path))

## Display one cell surface

In [ ]:
pv.set_jupyter_backend('trame') # Ensures interactive plots work in Jupyter

# Create a mapping from label to path
labels = [p.parent.parent.name for p in surface_paths]
path_map = {label: path for label, path in zip(labels, surface_paths)}

# Initialize a persistent plotter
plotter = pv.Plotter()
plotter.set_background("black")

def display_cell_surface(label):
    path = path_map[label]
    update_interactive_plotter(plotter, path)

# Dropdown menu
label_dropdown = widgets.Dropdown(
    options=labels, 
    value=labels[0] if labels else None, 
    description='Cell:'
)

# Link the dropdown to the update function
widgets.interactive_output(display_cell_surface, {'label': label_dropdown})

# Show the plotter once and display the controls
plotter.show()
display(label_dropdown)

## View surface in Paraview

1. Open Paraview
2. Import all .ply surfaces of interest
3. Apply
4. Select all surfaces -> Filters -> Alphabetical -> Group Datasets -> Apply